In [7]:
# =====================================================
# Saving Nakazato_2013 20 Msun models for bulkice_doumeki post processing
# =====================================================
import numpy as np
import h5py
import astropy.units as u
from snewpy.models.ccsn import Nakazato_2013
from snewpy.neutrino import Flavor, MixingParameters, MassHierarchy
from snewpy.flavor_transformation import AdiabaticMSW
import os

# =====================================================
# Step 1. Load valid Nakazato_2013 combinations
# =====================================================
param_combos = Nakazato_2013.get_param_combinations()

# Keep only 20 solar-mass models
param_combos_20 = [
    combo for combo in param_combos
    if np.isclose(combo["progenitor_mass"].to_value(u.solMass), 20.0)
]

print(f"Found {len(param_combos_20)} valid Nakazato_2013 20 Msun models.")

# =====================================================
# Step 2. Loop over each 20 Msun model
# =====================================================
for combo in param_combos_20:

    mass = combo["progenitor_mass"].to(u.solMass)
    revival_time = combo["revival_time"].to(u.ms)
    metallicity = combo["metallicity"]
    eos = combo["eos"]

    print(
        f"\n🚀 Processing Nakazato_2013 model: "
        f"{mass.value:.1f} M☉, t_rev={revival_time.value:.0f} ms, "
        f"z={metallicity}, eos={eos}"
    )

    # Initialize model
    model = Nakazato_2013(
        progenitor_mass=mass,
        revival_time=revival_time,
        metallicity=metallicity,
        eos=eos
    )

    # =====================================================
    # Step 3. Define flavor and extract time, flux
    # =====================================================
    flavor = Flavor.NU_E_BAR

    time = model.time.to(u.s).value
    flux = (model.luminosity[flavor] / 1e51).value  # foe/s

    t_min, t_max = time.min(), time.max()

    # =====================================================
    # Step 4. Define energy grid and resample time
    # =====================================================
    E = np.arange(0, 101, 2) * u.MeV
    time_sampled = np.arange(t_min, t_max, 0.01) * u.s

    model.interpolation = "nearest"

    # =====================================================
    # Step 5. Compute spectra: no osc, NMO, IMO
    # =====================================================

    # --- No oscillation
    spec_all_no = model.get_initial_spectra(time_sampled, E)
    spec_no_osc = spec_all_no[flavor].squeeze()

    # --- Normal hierarchy
    xnmo = AdiabaticMSW(MixingParameters(MassHierarchy.NORMAL))
    spec_all_nmo = model.get_transformed_spectra(time_sampled, E, xnmo)
    spec_nmo = spec_all_nmo[flavor].squeeze()

    # --- Inverted hierarchy
    ximo = AdiabaticMSW(MixingParameters(MassHierarchy.INVERTED))
    spec_all_imo = model.get_transformed_spectra(time_sampled, E, ximo)
    spec_imo = spec_all_imo[flavor].squeeze()

    # =====================================================
    # Step 6. Prepare output path and ensure overwrite
    # =====================================================
    outdir = "/Users/walu/icecube/snewpy_model_data/nakazato_2013"
    os.makedirs(outdir, exist_ok=True)

    z_str = str(metallicity).replace(".", "p")
    filename = os.path.join(
        outdir,
        f"Nakazato2013_{mass.value:.1f}Msun_"
        f"trev{revival_time.value:.0f}ms_z{z_str}_{eos}_"
        f"NU_E_BAR_with_osc.h5"
    )

    if os.path.exists(filename):
        os.remove(filename)
        print(f"⚠️  Existing file removed: {filename}")

    # =====================================================
    # Step 7. Write HDF5 file
    # =====================================================
    with h5py.File(filename, "w") as f:

        group_name = (
            f"models/Nakazato2013_{mass.value:.1f}M_"
            f"trev{revival_time.value:.0f}ms_z{z_str}_{eos}_NU_E_BAR"
        )

        g = f.create_group(group_name)

        # Metadata
        g.attrs["model_name"] = "Nakazato_2013"
        g.attrs["progenitor_mass_Msun"] = mass.value
        g.attrs["revival_time_ms"] = revival_time.value
        g.attrs["metallicity"] = metallicity
        g.attrs["eos"] = eos
        g.attrs["t_min_s"] = t_min
        g.attrs["t_max_s"] = t_max
        g.attrs["energy_bin_MeV"] = 2.0
        g.attrs["time_bin_s"] = 0.01
        g.attrs["flavor_name"] = "NU_E_BAR"

        # Core datasets
        g.create_dataset("time", data=time_sampled.value, dtype="f8", compression="gzip")
        g.create_dataset("energy", data=E.value, dtype="f8", compression="gzip")
        g.create_dataset("flux", data=flux_interp, dtype="f8", compression="gzip")

        # Spectra group
        sgrp = g.create_group("spectra")
        sgrp.create_dataset("no_osc", data=spec_no_osc, dtype="f8", compression="gzip")
        sgrp.create_dataset("nmo", data=spec_nmo, dtype="f8", compression="gzip")
        sgrp.create_dataset("imo", data=spec_imo, dtype="f8", compression="gzip")

    print(
        f"✅ Saved Nakazato_2013 {mass.value:.1f} M☉ "
        f"t_rev={revival_time.value:.0f} ms, z={metallicity}, eos={eos} → {filename}"
    )

Found 6 valid Nakazato_2013 20 Msun models.

🚀 Processing Nakazato_2013 model: 20.0 M☉, t_rev=100 ms, z=0.02, eos=shen


ValueError: too many values to unpack (expected 3)